# Phase 7 — Support-safe policy and offline evaluation

This notebook audits RecoverAI V2 without retraining XGBoost V1. It separates three questions:

1. Where does the frozen logging data support each intervention?
2. What do IPS and doubly robust estimators say using observed logged outcomes only?
3. How does the support-safe policy behave in the synthetic counterfactual environment?

These results must not be combined into a single causal claim.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "ml" / "reports" / "support_report.json").exists():
            return candidate
    raise FileNotFoundError("Run the Phase 7 report commands first")


ROOT = find_repo_root()
REPORTS = ROOT / "ml" / "reports"
support = json.loads((REPORTS / "support_report.json").read_text())
offline = json.loads((REPORTS / "offline_policy_metrics.json").read_text())
synthetic = json.loads((REPORTS / "synthetic_policy_evaluation.json").read_text())
decisions = pd.read_csv(REPORTS / "policy_decisions.csv")
context_support = pd.read_csv(REPORTS / "context_support.csv")

print("Frozen test decisions:", len(decisions))
print("Verified V1 artifacts:", len(support["frozen_artifacts_verified"]))

In [ ]:
pd.Series(support["threshold_derivation"], name="value").to_frame()

assert support["threshold_derivation"]["median_positive_cell_count"] == 22
assert support["support_thresholds"]["min_context_action_count"] == 20
assert support["support_thresholds"]["min_context_effective_sample_size"] == 8.0
assert support["counterfactual_outcomes_used"] is False

In [ ]:
action_rows = []
for action, values in support["actions"].items():
    action_rows.append({
        "action": action,
        "train_count": values["splits"]["train"]["count"],
        "validation_count": values["splits"]["validation"]["count"],
        "test_count": values["splits"]["test"]["count"],
        "observed_recovery_rate": values["observed_recovery_rate"],
        "mean_predicted_probability": values["mean_predicted_probability"],
    })
action_report = pd.DataFrame(action_rows).set_index("action")
action_report

In [ ]:
support_rates = pd.Series(
    support["test_candidate_support_rates"],
    name="supported_test_share",
).sort_values()
ax = support_rates.plot.barh(
    title="Frozen test cases with contextual action support",
    figsize=(8, 4),
)
ax.set(xlabel="Share of test payments", ylabel="Candidate intervention", xlim=(0, 1))
plt.tight_layout()
plt.show()

In [ ]:
context_summary = context_support.groupby("action").agg(
    observed_context_cells=("supported", "size"),
    supported_context_cells=("supported", "sum"),
    median_action_count=("action_count", "median"),
    median_effective_sample_size=("effective_sample_size", "median"),
)
context_summary

In [ ]:
print("Retry investigation")
display(pd.Series(support["retry_investigation"], name="value").to_frame())
print("\nEscalation investigation")
display(pd.Series(support["escalation_investigation"], name="value").to_frame())

In [ ]:
policy_summary = support["support_safe_policy"]
print("Selected actions:", policy_summary["selected_action_counts"])
print("Fallback rate:", policy_summary["fallback_rate"])
print("Reasons:", policy_summary["decision_reason_counts"])
assert policy_summary["unsupported_model_selection_count"] == 0

In [ ]:
offline_rows = []
for policy_name, values in offline["policies"].items():
    offline_rows.append({
        "policy": policy_name,
        "IPS": values["ips"],
        "SNIPS": values["self_normalized_ips"],
        "DR": values["doubly_robust"],
        "DR_CI_low": values["confidence_intervals_95"]["doubly_robust"][0],
        "DR_CI_high": values["confidence_intervals_95"]["doubly_robust"][1],
        "matching_rate": values["matching_rate"],
        "weight_ESS": values["importance_weight_effective_sample_size"],
    })
offline_metrics = pd.DataFrame(offline_rows).set_index("policy")
offline_metrics

In [ ]:
plot_data = offline_metrics.sort_values("DR")
errors = [
    plot_data["DR"] - plot_data["DR_CI_low"],
    plot_data["DR_CI_high"] - plot_data["DR"],
]
ax = plot_data["DR"].plot.barh(
    xerr=errors,
    figsize=(9, 5),
    title="Doubly robust recovery estimates with customer-bootstrap 95% CIs",
)
ax.set(xlabel="Estimated recovery rate", ylabel="Target policy", xlim=(0, 1))
plt.tight_layout()
plt.show()

In [ ]:
clipping_rows = []
for threshold, policies in offline["clipping_sensitivity"].items():
    for policy_name, values in policies.items():
        clipping_rows.append({
            "minimum_propensity": float(threshold),
            "policy": policy_name,
            "IPS": values["ips"],
            "DR": values["doubly_robust"],
            "weight_ESS": values["importance_weight_effective_sample_size"],
        })
clipping = pd.DataFrame(clipping_rows)
clipping.pivot(index="minimum_propensity", columns="policy", values="DR")

In [ ]:
original = pd.DataFrame(synthetic["original_environment"]).T
print("Original synthetic environment")
display(original)
print(
    "Sensitivity uplift range:",
    synthetic["minimum_v2_uplift_percentage_points"],
    "to",
    synthetic["maximum_v2_uplift_percentage_points"],
    "percentage points",
)
print(
    "V2 beats always-retry in",
    synthetic["scenarios_where_v2_beats_always_retry"],
    "of",
    synthetic["total_sensitivity_scenarios"],
    "perturbations",
)

In [ ]:
explanation_columns = [
    "payment_id",
    "raw_best_action",
    "selected_action",
    "selected_probability",
    "decision_margin",
    "fallback_used",
    "decision_reason",
    "candidate_actions",
]
decisions[explanation_columns].sample(5, random_state=42)

## Phase 7 conclusion

- V1 remains frozen and unchanged.
- V2 never makes a model-based selection without contextual support; manual escalation is the explicit safety fallback when no candidate is supported.
- Offline DR point estimates favor V2 over always-retry, but confidence intervals are wide and overlap. The result is directional, not statistically conclusive.
- In the original synthetic environment V2 is approximately tied with always-retry, and it wins only a subset of probability perturbations. The V1 headline uplift is therefore not robust enough for production claims.
- Counterfactual simulator outcomes are used only in the separately labeled synthetic evaluation, never in IPS/DR or policy selection.
- No execution integration is authorized by these results.